In [1]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

In [ ]:

data_folder = Path(r"C:\seem3650")


print("exist?：", data_folder.exists())
print("path：", data_folder.resolve())

exist?： True
path： C:\seem3650


In [3]:
month_list = pd.period_range(start="2023-01", end="2026-04", freq="M")
month_str_list = [m.strftime("%Y%m") for m in month_list]

print("total no. of month：", len(month_str_list))
print(month_str_list[:5], "...", month_str_list[-5:])

total no. of month： 40
['202301', '202302', '202303', '202304', '202305'] ... ['202512', '202601', '202602', '202603', '202604']


In [4]:
def process_aqhi_month(file_path, encoding="utf-8"):
    
    df = pd.read_csv(file_path, skiprows=7, encoding=encoding)
    
  
    df.columns = [str(c).strip() for c in df.columns]
    

    df["Hour"] = pd.to_numeric(df["Hour"], errors="coerce")
    df = df[df["Hour"].notna()].copy()
    
   
    df["Date"] = df["Date"].ffill()
    
   
    long_df = df.melt(
        id_vars=["Date", "Hour"],
        var_name="station",
        value_name="aqhi_raw"
    )
    
  
    long_df["has_asterisk"] = (
        long_df["aqhi_raw"]
        .astype(str)
        .str.contains(r"\*", regex=True, na=False)
    )
    
    
    long_df["aqhi"] = (
        long_df["aqhi_raw"]
        .astype(str)
        .str.strip()
        .str.replace("*", "", regex=False)
    )
    long_df["aqhi"] = long_df["aqhi"].replace(["", "nan", "None", "-"], pd.NA)
    long_df["aqhi"] = pd.to_numeric(long_df["aqhi"], errors="coerce")
    
    
    long_df["date"] = pd.to_datetime(long_df["Date"], errors="coerce")
    long_df["hour_index"] = pd.to_numeric(long_df["Hour"], errors="coerce")
    
    
    long_df["station"] = long_df["station"].astype(str).str.strip()
    
    
    long_df["source_file"] = os.path.basename(file_path)
    
    
    long_df = long_df[
        ["date", "hour_index", "station", "aqhi", "has_asterisk", "source_file"]
    ].copy()
    
    return long_df

In [5]:
def find_month_file(data_folder, yyyymm):
    
    
    
    candidate_1 = data_folder / f"{yyyymm}_Eng.csv"
    if candidate_1.exists():
        return candidate_1
    
    
    
    matched_files = sorted(data_folder.glob(f"*{yyyymm}*.csv"))
    if len(matched_files) > 0:
        return matched_files[0]
    
    
    return None

In [6]:
file_records = []

for yyyymm in month_str_list:
    file_path = find_month_file(data_folder, yyyymm)
    file_records.append({
        "yyyymm": yyyymm,
        "file_path": str(file_path) if file_path is not None else None,
        "found": file_path is not None
    })

file_check_df = pd.DataFrame(file_records)
file_check_df

,yyyymm,file_path,found
0,202301,C:\seem3650\202301_Eng.csv,True
1,202302,C:\seem3650\202302_Eng.csv,True
2,202303,C:\seem3650\202303_Eng.csv,True
3,202304,C:\seem3650\202304_Eng.csv,True
4,202305,C:\seem3650\202305_Eng.csv,True
5,202306,C:\seem3650\202306_Eng.csv,True
6,202307,C:\seem3650\202307_Eng.csv,True
7,202308,C:\seem3650\202308_Eng.csv,True
8,202309,C:\seem3650\202309_Eng.csv,True
9,202310,C:\seem3650\202310_Eng.csv,True


In [7]:
print("month?：", file_check_df["found"].sum())
print("less month?：", (~file_check_df["found"]).sum())

missing_months = file_check_df.loc[~file_check_df["found"], "yyyymm"].tolist()
print("less month：", missing_months)

month?： 39
less month?： 1
less month： ['202404']


In [8]:
all_months = []
read_log = []

for yyyymm in month_str_list:
    file_path = find_month_file(data_folder, yyyymm)
    
    if file_path is None:
        read_log.append({
            "yyyymm": yyyymm,
            "status": "missing_file",
            "file_path": None,
            "n_rows": None
        })
        continue
    
    try:
        temp = process_aqhi_month(file_path, encoding="utf-8")
        used_encoding = "utf-8"
    except UnicodeDecodeError:
        temp = process_aqhi_month(file_path, encoding="big5")
        used_encoding = "big5"
    
    temp["yyyymm"] = yyyymm
    all_months.append(temp)
    
    read_log.append({
        "yyyymm": yyyymm,
        "status": "ok",
        "file_path": str(file_path),
        "encoding": used_encoding,
        "n_rows": len(temp)
    })

read_log_df = pd.DataFrame(read_log)
read_log_df

,yyyymm,status,file_path,encoding,n_rows
0,202301,ok,C:\seem3650\202301_Eng.csv,utf-8,13392.0
1,202302,ok,C:\seem3650\202302_Eng.csv,utf-8,12096.0
2,202303,ok,C:\seem3650\202303_Eng.csv,utf-8,13392.0
3,202304,ok,C:\seem3650\202304_Eng.csv,utf-8,12960.0
4,202305,ok,C:\seem3650\202305_Eng.csv,utf-8,13392.0
5,202306,ok,C:\seem3650\202306_Eng.csv,utf-8,12960.0
6,202307,ok,C:\seem3650\202307_Eng.csv,utf-8,13392.0
7,202308,ok,C:\seem3650\202308_Eng.csv,utf-8,13392.0
8,202309,ok,C:\seem3650\202309_Eng.csv,utf-8,12960.0
9,202310,ok,C:\seem3650\202310_Eng.csv,utf-8,13392.0


In [9]:
aqhi_hourly = pd.concat(all_months, ignore_index=True)

aqhi_hourly = aqhi_hourly.drop_duplicates(
    subset=["date", "hour_index", "station"]
).copy()


aqhi_hourly = aqhi_hourly.sort_values(
    ["station", "date", "hour_index"]
).reset_index(drop=True)

aqhi_hourly.head(20)

,date,hour_index,station,aqhi,has_asterisk,source_file,yyyymm
0,2023-01-01,1.0,Causeway Bay,4.0,False,202301_Eng.csv,202301
1,2023-01-01,2.0,Causeway Bay,4.0,False,202301_Eng.csv,202301
2,2023-01-01,3.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
3,2023-01-01,4.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
4,2023-01-01,5.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
5,2023-01-01,6.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
6,2023-01-01,7.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
7,2023-01-01,8.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
8,2023-01-01,9.0,Causeway Bay,3.0,False,202301_Eng.csv,202301
9,2023-01-01,10.0,Causeway Bay,3.0,False,202301_Eng.csv,202301


In [10]:
print("hourly ：", len(aqhi_hourly))
print("date：", aqhi_hourly["date"].min(), "to", aqhi_hourly["date"].max())
print("station：", aqhi_hourly["station"].nunique())
print("AQHI less：", aqhi_hourly["aqhi"].isna().sum())
print("with *：", aqhi_hourly["has_asterisk"].sum())

hourly ： 512352
date： 2023-01-01 00:00:00 to 2026-04-30 00:00:00
station： 18
AQHI less： 935
with *： 12992


In [11]:
station_list = sorted(aqhi_hourly["station"].dropna().unique().tolist())
station_list

['Causeway Bay',
 'Central',
 'Central/Western',
 'Eastern',
 'Kwai Chung',
 'Kwun Tong',
 'Mong Kok',
 'North',
 'Sha Tin',
 'Sham Shui Po',
 'Southern',
 'Tai Po',
 'Tap Mun',
 'Tseung Kwan O',
 'Tsuen Wan',
 'Tuen Mun',
 'Tung Chung',
 'Yuen Long']

In [12]:
daily_hour_counts = (
    aqhi_hourly
    .groupby(["date", "station"])
    .size()
    .reset_index(name="n_rows")
)

daily_hour_counts["n_rows"].value_counts().sort_index()

n_rows
24    21348
Name: count, dtype: int64

In [13]:
aqhi_daily = (
    aqhi_hourly
    .groupby(["date", "station"], as_index=False)
    .agg(
        daily_max_aqhi=("aqhi", "max"),
        daily_mean_aqhi=("aqhi", "mean"),
        hourly_count=("aqhi", "count"),
        starred_hours=("has_asterisk", "sum")
    )
)

aqhi_daily = aqhi_daily.sort_values(["station", "date"]).reset_index(drop=True)
aqhi_daily.head(20)

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours
0,2023-01-01,Causeway Bay,4.0,3.541667,24,0
1,2023-01-02,Causeway Bay,5.0,3.625000,24,0
2,2023-01-03,Causeway Bay,5.0,3.416667,24,0
3,2023-01-04,Causeway Bay,5.0,3.791667,24,0
4,2023-01-05,Causeway Bay,7.0,4.666667,24,0
5,2023-01-06,Causeway Bay,5.0,4.166667,24,0
6,2023-01-07,Causeway Bay,5.0,4.083333,24,0
7,2023-01-08,Causeway Bay,5.0,4.375000,24,0
8,2023-01-09,Causeway Bay,5.0,4.333333,24,0
9,2023-01-10,Causeway Bay,3.0,2.791667,24,1


In [14]:
print("daily ：", len(aqhi_daily))
print("date：", aqhi_daily["date"].min(), "to", aqhi_daily["date"].max())
print("station：", aqhi_daily["station"].nunique())
print("daily_max_aqhi ：", aqhi_daily["daily_max_aqhi"].isna().sum())

daily ： 21348
date： 2023-01-01 00:00:00 to 2026-04-30 00:00:00
station： 18
daily_max_aqhi ： 1


In [15]:
aqhi_daily["hourly_count"].value_counts().sort_index()

hourly_count
0         1
5         1
10        1
11        1
13        2
14        5
15       18
17        4
18        5
19        9
20       28
21       15
22      104
23      163
24    20991
Name: count, dtype: int64

In [16]:
aqhi_daily["lag1_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"].shift(1)
)

aqhi_daily["lag2_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"].shift(2)
)

aqhi_daily["lag3_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"].shift(3)
)

aqhi_daily["lag7_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"].shift(7)
)

In [17]:
aqhi_daily["rolling3_mean_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

aqhi_daily["rolling7_mean_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

aqhi_daily["rolling14_mean_aqhi"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
)

In [18]:
aqhi_daily["target_next_day"] = (
    aqhi_daily.groupby("station")["daily_max_aqhi"].shift(-1)
)

In [19]:
aqhi_daily["high_risk_next_day"] = (
    aqhi_daily["target_next_day"] >= 7
).astype("float")


aqhi_daily.loc[
    aqhi_daily["target_next_day"].isna(),
    "high_risk_next_day"
] = np.nan

In [20]:
aqhi_daily["year"] = aqhi_daily["date"].dt.year
aqhi_daily["month"] = aqhi_daily["date"].dt.month
aqhi_daily["day"] = aqhi_daily["date"].dt.day
aqhi_daily["day_of_week"] = aqhi_daily["date"].dt.dayofweek
aqhi_daily["is_weekend"] = aqhi_daily["day_of_week"].isin([5, 6]).astype(int)

In [21]:
aqhi_hourly[
    (aqhi_hourly["date"] == "2023-09-02") &
    (aqhi_hourly["station"] == "Eastern")
].sort_values("hour_index")

,date,hour_index,station,aqhi,has_asterisk,source_file,yyyymm
91248,2023-09-02,1.0,Eastern,3.0,False,202309_Eng.csv,202309
91249,2023-09-02,2.0,Eastern,3.0,False,202309_Eng.csv,202309
91250,2023-09-02,3.0,Eastern,3.0,False,202309_Eng.csv,202309
91251,2023-09-02,4.0,Eastern,3.0,False,202309_Eng.csv,202309
91252,2023-09-02,5.0,Eastern,3.0,False,202309_Eng.csv,202309
91253,2023-09-02,6.0,Eastern,3.0,False,202309_Eng.csv,202309
91254,2023-09-02,7.0,Eastern,3.0,False,202309_Eng.csv,202309
91255,2023-09-02,8.0,Eastern,3.0,False,202309_Eng.csv,202309
91256,2023-09-02,9.0,Eastern,3.0,False,202309_Eng.csv,202309
91257,2023-09-02,10.0,Eastern,3.0,False,202309_Eng.csv,202309


In [22]:
aqhi_daily[aqhi_daily["daily_max_aqhi"].isna()]

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours,lag1_aqhi,lag2_aqhi,lag3_aqhi,lag7_aqhi,rolling3_mean_aqhi,rolling7_mean_aqhi,rolling14_mean_aqhi,target_next_day,high_risk_next_day,year,month,day,day_of_week,is_weekend
14476,2023-09-02,Tap Mun,NaN,NaN,0,24,3.0,3.0,4.0,3.0,3.333333,3.285714,3.071429,2.0,0.0,2023,9,2,5,1


In [23]:
aqhi_daily.head(20)

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours,lag1_aqhi,lag2_aqhi,lag3_aqhi,lag7_aqhi,rolling3_mean_aqhi,rolling7_mean_aqhi,rolling14_mean_aqhi,target_next_day,high_risk_next_day,year,month,day,day_of_week,is_weekend
0,2023-01-01,Causeway Bay,4.0,3.541667,24,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0.0,2023,1,1,6,1
1,2023-01-02,Causeway Bay,5.0,3.625000,24,0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0.0,2023,1,2,0,0
2,2023-01-03,Causeway Bay,5.0,3.416667,24,0,5.0,4.0,NaN,NaN,NaN,NaN,NaN,5.0,0.0,2023,1,3,1,0
3,2023-01-04,Causeway Bay,5.0,3.791667,24,0,5.0,5.0,4.0,NaN,4.666667,NaN,NaN,7.0,1.0,2023,1,4,2,0
4,2023-01-05,Causeway Bay,7.0,4.666667,24,0,5.0,5.0,5.0,NaN,5.000000,NaN,NaN,5.0,0.0,2023,1,5,3,0
5,2023-01-06,Causeway Bay,5.0,4.166667,24,0,7.0,5.0,5.0,NaN,5.666667,NaN,NaN,5.0,0.0,2023,1,6,4,0
6,2023-01-07,Causeway Bay,5.0,4.083333,24,0,5.0,7.0,5.0,NaN,5.666667,NaN,NaN,5.0,0.0,2023,1,7,5,1
7,2023-01-08,Causeway Bay,5.0,4.375000,24,0,5.0,5.0,7.0,4.0,5.666667,5.142857,NaN,5.0,0.0,2023,1,8,6,1
8,2023-01-09,Causeway Bay,5.0,4.333333,24,0,5.0,5.0,5.0,5.0,5.000000,5.285714,NaN,3.0,0.0,2023,1,9,0,0
9,2023-01-10,Causeway Bay,3.0,2.791667,24,1,5.0,5.0,5.0,5.0,5.000000,5.285714,NaN,4.0,0.0,2023,1,10,1,0


In [24]:
model_df = aqhi_daily.dropna(subset=["target_next_day"]).copy()
model_df.head(20)

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours,lag1_aqhi,lag2_aqhi,lag3_aqhi,lag7_aqhi,rolling3_mean_aqhi,rolling7_mean_aqhi,rolling14_mean_aqhi,target_next_day,high_risk_next_day,year,month,day,day_of_week,is_weekend
0,2023-01-01,Causeway Bay,4.0,3.541667,24,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0.0,2023,1,1,6,1
1,2023-01-02,Causeway Bay,5.0,3.625000,24,0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0.0,2023,1,2,0,0
2,2023-01-03,Causeway Bay,5.0,3.416667,24,0,5.0,4.0,NaN,NaN,NaN,NaN,NaN,5.0,0.0,2023,1,3,1,0
3,2023-01-04,Causeway Bay,5.0,3.791667,24,0,5.0,5.0,4.0,NaN,4.666667,NaN,NaN,7.0,1.0,2023,1,4,2,0
4,2023-01-05,Causeway Bay,7.0,4.666667,24,0,5.0,5.0,5.0,NaN,5.000000,NaN,NaN,5.0,0.0,2023,1,5,3,0
5,2023-01-06,Causeway Bay,5.0,4.166667,24,0,7.0,5.0,5.0,NaN,5.666667,NaN,NaN,5.0,0.0,2023,1,6,4,0
6,2023-01-07,Causeway Bay,5.0,4.083333,24,0,5.0,7.0,5.0,NaN,5.666667,NaN,NaN,5.0,0.0,2023,1,7,5,1
7,2023-01-08,Causeway Bay,5.0,4.375000,24,0,5.0,5.0,7.0,4.0,5.666667,5.142857,NaN,5.0,0.0,2023,1,8,6,1
8,2023-01-09,Causeway Bay,5.0,4.333333,24,0,5.0,5.0,5.0,5.0,5.000000,5.285714,NaN,3.0,0.0,2023,1,9,0,0
9,2023-01-10,Causeway Bay,3.0,2.791667,24,1,5.0,5.0,5.0,5.0,5.000000,5.285714,NaN,4.0,0.0,2023,1,10,1,0


In [25]:
model_df_strict = aqhi_daily.copy()

model_df_strict = model_df_strict[model_df_strict["hourly_count"] == 24]

model_df_strict = model_df_strict.dropna(subset=[
    "lag1_aqhi",
    "lag2_aqhi",
    "lag3_aqhi",
    "lag7_aqhi",
    "rolling3_mean_aqhi",
    "rolling7_mean_aqhi",
    "rolling14_mean_aqhi",
    "target_next_day"
]).copy()

model_df_strict.head(20)

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours,lag1_aqhi,lag2_aqhi,lag3_aqhi,lag7_aqhi,rolling3_mean_aqhi,rolling7_mean_aqhi,rolling14_mean_aqhi,target_next_day,high_risk_next_day,year,month,day,day_of_week,is_weekend
14,2023-01-15,Causeway Bay,3.0,2.541667,24,0,3.0,4.0,4.0,5.0,3.666667,4.000000,4.571429,3.0,0.0,2023,1,15,6,1
15,2023-01-16,Causeway Bay,3.0,2.666667,24,7,3.0,3.0,4.0,5.0,3.333333,3.714286,4.500000,3.0,0.0,2023,1,16,0,0
16,2023-01-17,Causeway Bay,3.0,2.750000,24,8,3.0,3.0,3.0,3.0,3.000000,3.428571,4.357143,5.0,0.0,2023,1,17,1,0
17,2023-01-18,Causeway Bay,5.0,3.583333,24,4,3.0,3.0,3.0,4.0,3.000000,3.428571,4.214286,5.0,0.0,2023,1,18,2,0
18,2023-01-19,Causeway Bay,5.0,4.041667,24,0,5.0,3.0,3.0,4.0,3.666667,3.571429,4.214286,4.0,0.0,2023,1,19,3,0
19,2023-01-20,Causeway Bay,4.0,3.791667,24,0,5.0,5.0,3.0,4.0,4.333333,3.714286,4.071429,4.0,0.0,2023,1,20,4,0
20,2023-01-21,Causeway Bay,4.0,3.958333,24,0,4.0,5.0,5.0,3.0,4.666667,3.714286,4.000000,4.0,0.0,2023,1,21,5,1
21,2023-01-22,Causeway Bay,4.0,3.208333,24,0,4.0,4.0,5.0,3.0,4.333333,3.857143,3.928571,4.0,0.0,2023,1,22,6,1
22,2023-01-23,Causeway Bay,4.0,3.333333,24,0,4.0,4.0,4.0,3.0,4.000000,4.000000,3.857143,4.0,0.0,2023,1,23,0,0
23,2023-01-24,Causeway Bay,4.0,3.291667,24,1,4.0,4.0,4.0,3.0,4.000000,4.142857,3.785714,5.0,0.0,2023,1,24,1,0


In [26]:
print("target_next_day 描述統計：")
print(model_df["target_next_day"].describe())

print("\nhigh_risk_next_day 分布：")
print(model_df["high_risk_next_day"].value_counts(dropna=False))

target_next_day 描述統計：
count    21329.000000
mean         4.256552
std          1.478309
min          1.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         10.000000
Name: target_next_day, dtype: float64

high_risk_next_day 分布：
high_risk_next_day
0.0    19885
1.0     1444
Name: count, dtype: int64


In [27]:
model_df.isna().sum().sort_values(ascending=False)
model_df[model_df["daily_max_aqhi"].isna()]

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours,lag1_aqhi,lag2_aqhi,lag3_aqhi,lag7_aqhi,rolling3_mean_aqhi,rolling7_mean_aqhi,rolling14_mean_aqhi,target_next_day,high_risk_next_day,year,month,day,day_of_week,is_weekend
14476,2023-09-02,Tap Mun,NaN,NaN,0,24,3.0,3.0,4.0,3.0,3.333333,3.285714,3.071429,2.0,0.0,2023,9,2,5,1


In [28]:
output_folder = Path("data/processed_aqhi")
output_folder.mkdir(parents=True, exist_ok=True)

aqhi_hourly.to_csv(output_folder / "aqhi_hourly_202301_202604.csv", index=False, encoding="utf-8-sig")
aqhi_daily.to_csv(output_folder / "aqhi_daily_202301_202604.csv", index=False, encoding="utf-8-sig")
model_df.to_csv(output_folder / "aqhi_model_df_202301_202604.csv", index=False, encoding="utf-8-sig")
model_df_strict.to_csv(output_folder / "aqhi_model_df_strict_202301_202604.csv", index=False, encoding="utf-8-sig")

print("output：", output_folder.resolve())

output： C:\seem3650\data\processed_aqhi


In [ ]:
print("finish")
print("hourly：", len(aqhi_hourly))
print("daily ：", len(aqhi_daily))
print("model_df ：", len(model_df))
print("model_df_strict ：", len(model_df_strict))
print("date：", aqhi_daily['date'].min(), "to", aqhi_daily['date'].max())
print("station：", aqhi_daily['station'].nunique())

finish
hourly： 512352
daily ： 21348
model_df ： 21329
model_df_strict ： 20712
date： 2023-01-01 00:00:00 到 2026-04-30 00:00:00
station： 18
